# Credit Risk ML Phase: cleaned notebook

Consolidated from the working notebook. Duplicates, abandoned versions and one buggy cell were dropped.

**How to run**
- Needs `staging_application_train` and the materialized view `mv_customer_payment_features` in the `home_credit` database (built by `04_views/02_materialized_payment_features.sql`).
- **Part 1 and Part 2 are a chain: run their cells in order, top to bottom** (they share variables).
- **Parts 3 onward are self-contained**: each cell has its own imports and data load, so each can be run alone after a kernel restart.
- Install once, in a separate cell: `%pip install lightgbm xgboost shap` (then restart the kernel).
- Every model uses the same split: `train_test_split(test_size=0.2, stratify=target, random_state=42)`.

**Dropped on purpose:** the first self-contained LightGBM cell (used `scale_pos_weight` with early stopping, so it stopped after 1 tree and scored AUC 0.715), the duplicate importance cell, the unused setup cell, and the raw-CSV loaders (already in `02_data_load/`).

## Part 1. Feature screening (Information Value)

**Known issue, now checked:** the first cell computes IV on **all 307,511 rows**, including the rows later used as the test set, so test data could have influenced which 19 features were kept. The second cell recomputes IV on the train rows only (it is self-contained and can be run alone). Result: no IV moved by more than 0.0035, and none of the 19 selected features changed side of the 0.02 line, so the selection did not change. The numbers are in the markdown cell right after it.

The IV function's smoothing (`+0.5` in numerator and denominator) is a quick approximation, not the textbook formula.


In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

engine = create_engine("postgresql+psycopg2://postgres@localhost:5432/home_credit")
df = pd.read_sql("SELECT * FROM staging_application_train", engine)

def calculate_iv(df, feature, target, bins=10):
    data = df[[feature, target]].copy()
    if data[feature].dtype in ['float64', 'int64'] and data[feature].nunique() > bins:
        data['bin'] = pd.qcut(data[feature], q=bins, duplicates='drop')
    else:
        data['bin'] = data[feature].astype(str)
    
    data['bin'] = data['bin'].astype(str)
    data.loc[data[feature].isna(), 'bin'] = 'MISSING'

    grouped = data.groupby('bin')[target].agg(['count', 'sum'])
    grouped.columns = ['total', 'bad']
    grouped['good'] = grouped['total'] - grouped['bad']

    total_bad = grouped['bad'].sum()
    total_good = grouped['good'].sum()

    grouped['bad_rate'] = (grouped['bad'] + 0.5) / (total_bad + 0.5)
    grouped['good_rate'] = (grouped['good'] + 0.5) / (total_good + 0.5)
    grouped['woe'] = np.log(grouped['good_rate'] / grouped['bad_rate'])
    grouped['iv'] = (grouped['good_rate'] - grouped['bad_rate']) * grouped['woe']

    return grouped['iv'].sum()

results = []
exclude_cols = ['sk_id_curr', 'target']

for col in df.columns:
    if col in exclude_cols:
        continue
    missing_pct = round(df[col].isna().mean() * 100, 2)
    try:
        corr = round(df[col].corr(df['target']), 4) if df[col].dtype in ['float64', 'int64'] else None
    except Exception:
        corr = None
    try:
        iv = round(calculate_iv(df, col, 'target'), 4)
    except Exception:
        iv = None
    results.append({'column': col, 'missing_pct': missing_pct, 'correlation': corr, 'iv': iv})

screening = pd.DataFrame(results).sort_values('iv', ascending=False)
print(screening.to_string())

screening.to_csv('feature_screening_results.csv', index=False)
print("\nSaved to feature_screening_results.csv")

In [ ]:
# ===== Train-only IV check (self-contained: can be run alone) =====
import os
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split

engine = create_engine("postgresql+psycopg2://postgres@localhost:5432/home_credit")
df = pd.read_sql("SELECT * FROM staging_application_train", engine)

def calculate_iv(data_in, feature, target, bins=10):
    data = data_in[[feature, target]].copy()
    if data[feature].dtype in ['float64', 'int64'] and data[feature].nunique() > bins:
        data['bin'] = pd.qcut(data[feature], q=bins, duplicates='drop')
    else:
        data['bin'] = data[feature].astype(str)
    data['bin'] = data['bin'].astype(str)
    data.loc[data[feature].isna(), 'bin'] = 'MISSING'

    g = data.groupby('bin')[target].agg(['count', 'sum'])
    g.columns = ['total', 'bad']
    g['good'] = g['total'] - g['bad']
    g['bad_rate'] = (g['bad'] + 0.5) / (g['bad'].sum() + 0.5)
    g['good_rate'] = (g['good'] + 0.5) / (g['good'].sum() + 0.5)
    g['woe'] = np.log(g['good_rate'] / g['bad_rate'])
    return ((g['good_rate'] - g['bad_rate']) * g['woe']).sum()

selected_19 = [
    'ext_source_1', 'ext_source_2', 'ext_source_3', 'days_employed',
    'amt_goods_price', 'days_birth', 'occupation_type', 'organization_type',
    'name_income_type', 'name_education_type', 'region_rating_client',
    'days_last_phone_change', 'amt_credit', 'code_gender',
    'amt_req_credit_bureau_year', 'def_30_cnt_social_circle',
    'name_family_status', 'name_housing_type', 'flag_document_3'
]

# Same split as every model
idx_train, idx_test = train_test_split(df.index, test_size=0.2, stratify=df['target'], random_state=42)
df_tr = df.loc[idx_train]

cols = [c for c in df.columns if c not in ['sk_id_curr', 'target']]

def iv_series(frame):
    out = {}
    for c in cols:
        try:
            out[c] = round(calculate_iv(frame, c, 'target'), 4)
        except Exception:
            out[c] = np.nan
    return pd.Series(out)

iv_train = iv_series(df_tr)

# Reuse the all-rows result from the cell above if its file is still in the working folder
if os.path.exists('feature_screening_results.csv'):
    iv_full = pd.read_csv('feature_screening_results.csv').set_index('column')['iv']
else:
    iv_full = iv_series(df)

cmp = pd.DataFrame({'iv_train': iv_train, 'iv_full': iv_full})
cmp['diff'] = (cmp['iv_train'] - cmp['iv_full']).round(4)

print("The 19 selected features: IV on train rows vs all rows")
print(cmp.loc[selected_19].sort_values('iv_train', ascending=False).to_string())

print("\nOf the 19, IV below 0.02 on train rows:")
print(cmp.loc[selected_19].query('iv_train < 0.02')[['iv_train', 'iv_full']])

print("\nIV >= 0.02 on train rows but NOT in the 19:")
print(cmp[(cmp['iv_train'] >= 0.02) & (~cmp.index.isin(selected_19))]
      .sort_values('iv_train', ascending=False)[['iv_train', 'iv_full']].to_string())

print("\nLargest train-vs-all-rows IV difference across ALL columns:", cmp['diff'].abs().max())


**Result of the train-only run** (copied from the printed output)

- Largest difference between train-only IV and all-rows IV, across all columns: **0.0035** (`ext_source_3`: 0.3293 on all rows, 0.3258 on train).
- None of the 19 selected features changed side of the 0.02 line. `name_family_status` (0.0230) and `flag_document_3` (0.0291) stay above it.
- Three of the 19 are below 0.02 on both versions: `amt_req_credit_bureau_year` (0.0174), `name_housing_type` (0.0146) and `def_30_cnt_social_circle` (0.0144). They were kept on domain grounds, so "IV >= 0.02" was a guide here, not a strict rule.
- 40 columns outside the 19 clear 0.02 on train rows: 31 are the building/apartment family, and 9 are not (`region_rating_client_w_city`, `days_id_publish`, `region_population_relative`, `reg_city_not_work_city`, `flag_emp_phone`, `days_registration`, `amt_annuity`, `reg_city_not_live_city`, `own_car_age`).
- Conclusion: computing IV on all rows did not change the selection. The bigger caveat is that the selection was looser than a plain cutoff. The all-columns logistic and the tree models do not depend on it.


## Part 2. Split, preprocessing and logistic regression on the 19 features (run in order)

Cells: split, preprocessing, baseline, coefficients, collinearity check, ratio vs difference tests, then payment-history features. Results: baseline test AUC 0.7460; ratio version 0.7462; with payment features 0.7516.

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split

engine = create_engine("postgresql+psycopg2://postgres@localhost:5432/home_credit")

selected_features = [
    'ext_source_1', 'ext_source_2', 'ext_source_3', 'days_employed',
    'amt_goods_price', 'days_birth', 'occupation_type', 'organization_type',
    'name_income_type', 'name_education_type', 'region_rating_client',
    'days_last_phone_change', 'amt_credit', 'code_gender',
    'amt_req_credit_bureau_year', 'def_30_cnt_social_circle',
    'name_family_status', 'name_housing_type', 'flag_document_3'
]

query = f"SELECT {', '.join(selected_features)}, target FROM staging_application_train"
df = pd.read_sql(query, engine)

# Fix known DAYS_EMPLOYED anomaly (365243 placeholder = not employed, not a real value)
df['days_employed'] = df['days_employed'].replace(365243, np.nan)

# Convert days_birth to actual age in years (more interpretable)
df['age_years'] = -df['days_birth'] / 365
df = df.drop(columns=['days_birth'])

X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train default rate:", round(y_train.mean() * 100, 2), "%")
print("Test default rate:", round(y_test.mean() * 100, 2), "%")

In [ ]:
# --- Preprocessing v2 (data prep, not analysis) ---
# Re-run the split cell first so X_train / X_test are fresh (unprocessed)

num_cols = X_train.select_dtypes(include='number').columns.tolist()
cat_cols = X_train.select_dtypes(exclude='number').columns.tolist()

X_train = X_train.copy()
X_test = X_test.copy()

# 1) Missing-indicator flags for the three heavily-missing columns
for c in ['ext_source_1', 'ext_source_3', 'days_employed']:
    X_train[c + '_missing'] = X_train[c].isna().astype(int)
    X_test[c + '_missing'] = X_test[c].isna().astype(int)

# 2) Median imputation: medians come from TRAIN only
medians = X_train[num_cols].median()
X_train[num_cols] = X_train[num_cols].fillna(medians)
X_test[num_cols] = X_test[num_cols].fillna(medians)

# 3) Categoricals: fill blanks with "Missing", then one-hot encode
X_train[cat_cols] = X_train[cat_cols].fillna('Missing')
X_test[cat_cols] = X_test[cat_cols].fillna('Missing')

X_train = pd.get_dummies(X_train, columns=cat_cols)
X_test = pd.get_dummies(X_test, columns=cat_cols)

# 4) Make sure test has exactly the same columns as train
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Any blanks left in train?", X_train.isna().sum().sum())
print("Any blanks left in test?", X_test.isna().sum().sum())
print("Same columns in both?", list(X_train.columns) == list(X_test.columns))


In [ ]:
# --- Logistic Regression baseline (real modeling step) ---
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

# 1) Scale: fit on TRAIN only, apply to both
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# 2) Train
lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train_s, y_train)

# 3) Predicted default probabilities
train_prob = lr.predict_proba(X_train_s)[:, 1]
test_prob = lr.predict_proba(X_test_s)[:, 1]

# 4) Metrics
train_auc = roc_auc_score(y_train, train_prob)
test_auc = roc_auc_score(y_test, test_prob)
fpr, tpr, _ = roc_curve(y_test, test_prob)
ks = (tpr - fpr).max()

print("Train AUC:", round(train_auc, 4))
print("Test AUC :", round(test_auc, 4))
print("Test Gini:", round(2 * test_auc - 1, 4))
print("Test KS  :", round(ks, 4))


In [ ]:
# --- Which features drive the model? (real analysis) ---
coef = pd.Series(lr.coef_[0], index=X_train.columns).sort_values()

print("Top 10 that LOWER default risk:")
print(coef.head(10).round(3))
print("\nTop 10 that RAISE default risk:")
print(coef.tail(10).round(3))

In [ ]:
print(X_train[['amt_credit', 'amt_goods_price']].corr())

In [ ]:
# --- Ratio feature + retrain (real modeling step) ---
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

def add_ratio(df):
    df = df.copy()
    df['credit_to_goods'] = df['amt_credit'] / df['amt_goods_price'].replace(0, np.nan)
    return df

Xtr = add_ratio(X_train)
Xte = add_ratio(X_test)

# Fill any blanks in the ratio using TRAIN median only
med = Xtr['credit_to_goods'].median()
Xtr['credit_to_goods'] = Xtr['credit_to_goods'].fillna(med)
Xte['credit_to_goods'] = Xte['credit_to_goods'].fillna(med)

def run(cols_to_drop, label):
    a = Xtr.drop(columns=cols_to_drop)
    b = Xte.drop(columns=cols_to_drop)
    sc = StandardScaler()
    a_s = sc.fit_transform(a)
    b_s = sc.transform(b)
    m = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    m.fit(a_s, y_train)
    tr = roc_auc_score(y_train, m.predict_proba(a_s)[:, 1])
    p = m.predict_proba(b_s)[:, 1]
    te = roc_auc_score(y_test, p)
    fpr, tpr, _ = roc_curve(y_test, p)
    print(f"{label}: Train AUC {tr:.4f} | Test AUC {te:.4f} | "
          f"Gini {2*te-1:.4f} | KS {(tpr-fpr).max():.4f}")
    return pd.Series(m.coef_[0], index=a.columns)

# Version A: ratio replaces both amount columns
coef_a = run(['amt_credit', 'amt_goods_price'], "A (ratio only)      ")

# Version B: ratio + keep credit amount for size
coef_b = run(['amt_goods_price'], "B (ratio + credit)  ")

print("\nVersion B weights for the amount features:")
print(coef_b[['amt_credit', 'credit_to_goods']].round(3))

In [ ]:
# --- Difference feature, compare against ratio (real modeling step) ---
Xtr['credit_minus_goods'] = Xtr['amt_credit'] - Xtr['amt_goods_price']
Xte['credit_minus_goods'] = Xte['amt_credit'] - Xte['amt_goods_price']

# Version C: difference replaces both amount columns
coef_c = run(['amt_credit', 'amt_goods_price', 'credit_to_goods'], "C (difference only) ")

# Version D: ratio AND difference together
coef_d = run(['amt_credit', 'amt_goods_price'], "D (ratio + diff)    ")

In [ ]:
# --- Add payment-history features + retrain (real modeling step) ---
mv = pd.read_sql("SELECT * FROM mv_customer_payment_features", engine).set_index('sk_id_curr')
ids = pd.read_sql("SELECT sk_id_curr, target FROM staging_application_train", engine)

# Safety check: row order must match the modeling data
print("Row order matches?", len(ids) == len(df) and (ids['target'].values == df['target'].values).all())

def attach(X):
    X = X.copy()
    # ratio feature (Version A): replaces both amount columns
    X['credit_to_goods'] = X['amt_credit'] / X['amt_goods_price'].replace(0, np.nan)
    X = X.drop(columns=['amt_credit', 'amt_goods_price'])
    # payment features, matched by customer ID
    P = mv.reindex(ids.loc[X.index, 'sk_id_curr'].values)
    P.index = X.index
    return pd.concat([X, P], axis=1)

Xtr3, Xte3 = attach(X_train), attach(X_test)

# Fill blanks using TRAIN medians only (flags already mark who had no history)
med3 = Xtr3.median()
Xtr3 = Xtr3.fillna(med3)
Xte3 = Xte3.fillna(med3)

sc = StandardScaler()
a_s = sc.fit_transform(Xtr3)
b_s = sc.transform(Xte3)

m = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
m.fit(a_s, y_train)

tr = roc_auc_score(y_train, m.predict_proba(a_s)[:, 1])
p = m.predict_proba(b_s)[:, 1]
te = roc_auc_score(y_test, p)
fpr, tpr, _ = roc_curve(y_test, p)
print(f"With payment history: Train AUC {tr:.4f} | Test AUC {te:.4f} | Gini {2*te-1:.4f} | KS {(tpr-fpr).max():.4f}")
print("Baseline was:         Test AUC 0.7462 | KS 0.3669")

coef = pd.Series(m.coef_[0], index=Xtr3.columns)
print("\nPayment-feature weights:")
print(coef[mv.columns].round(3))

## Part 3. Logistic regression on ALL columns + payment history (self-contained)

Test AUC 0.7558, KS 0.3843. Separates the effect of the feature set from the effect of the model type.

In [ ]:
# ===== Logistic regression on ALL columns + payment history (self-contained) =====
import time
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

engine = create_engine("postgresql+psycopg2://postgres@localhost:5432/home_credit")

# 1) Load everything, same cleanup as before
full = pd.read_sql("SELECT * FROM staging_application_train", engine)
full['days_employed'] = full['days_employed'].replace(365243, np.nan)

# 2) Same train/test rows as every other model
idx_train, idx_test = train_test_split(
    full.index, test_size=0.2, stratify=full['target'], random_state=42
)

# 3) Payment-history features
mv = pd.read_sql("SELECT * FROM mv_customer_payment_features", engine)
full = full.merge(mv, on='sk_id_curr', how='left')

# 4) Credit/goods ratio (replaces the near-duplicate goods price)
full['credit_to_goods'] = full['amt_credit'] / full['amt_goods_price'].replace(0, np.nan)
full = full.drop(columns=['amt_goods_price'])

feat = [c for c in full.columns if c not in ['sk_id_curr', 'target']]
Xa, ya = full.loc[idx_train, feat], full.loc[idx_train, 'target']
Xt, yt = full.loc[idx_test, feat], full.loc[idx_test, 'target']

num_cols = Xa.select_dtypes(include='number').columns.tolist()
cat_cols = Xa.select_dtypes(exclude='number').columns.tolist()

# 5) Preprocessing (everything is fit on TRAIN only inside the pipeline)
pre = ColumnTransformer([
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median', add_indicator=True)),  # adds "was missing" flags
        ('sc', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='constant', fill_value='Missing')),
        ('oh', OneHotEncoder(handle_unknown='ignore', min_frequency=100, sparse_output=False))
    ]), cat_cols),
])

model = Pipeline([
    ('pre', pre),
    ('lr', LogisticRegression(C=0.1, class_weight='balanced', max_iter=2000, random_state=42))
])

t0 = time.time()
model.fit(Xa, ya)
train_time = time.time() - t0

# 6) Evaluate
tr = roc_auc_score(ya, model.predict_proba(Xa)[:, 1])
p = model.predict_proba(Xt)[:, 1]
te = roc_auc_score(yt, p)
fpr, tpr, _ = roc_curve(yt, p)

print("Train rows:", len(Xa), "| Test rows:", len(Xt))
print("Model inputs after encoding:", model.named_steps['pre'].transform(Xa.head(5)).shape[1])
print(f"Training time: {train_time:.1f} sec")
print(f"Train AUC {tr:.4f} | Test AUC {te:.4f} | Gini {2*te-1:.4f} | KS {(tpr-fpr).max():.4f}")
print("Logistic (19 feats + payment): Test AUC 0.7516 | KS 0.3754")
print("LightGBM (all columns)       : Test AUC 0.7678 | KS 0.3982")

## Part 4. LightGBM, untuned (self-contained)

AUC-only early stopping, no class weighting. Test AUC 0.7678, KS 0.3982. Prints the top 20 features by gain.

In [ ]:
# ===== COMPLETE SELF-CONTAINED CELL: run this alone =====
import time
import pandas as pd
import numpy as np
import lightgbm as lgb
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

engine = create_engine("postgresql+psycopg2://postgres@localhost:5432/home_credit")

# 1) Load all columns from the staging table
full = pd.read_sql("SELECT * FROM staging_application_train", engine)
full['days_employed'] = full['days_employed'].replace(365243, np.nan)

# 2) Same train/test rows as the logistic regression
idx_train, idx_test = train_test_split(
    full.index, test_size=0.2, stratify=full['target'], random_state=42
)

# 3) Payment-history features from the materialized view
mv = pd.read_sql("SELECT * FROM mv_customer_payment_features", engine)
full = full.merge(mv, on='sk_id_curr', how='left')

# 4) Text columns -> category dtype
for c in full.select_dtypes(include='object').columns:
    full[c] = full[c].astype('category')

# 5) Features / target
feat = [c for c in full.columns if c not in ['sk_id_curr', 'target']]
Xa, ya = full.loc[idx_train, feat], full.loc[idx_train, 'target']
Xt, yt = full.loc[idx_test, feat], full.loc[idx_test, 'target']

# 6) Validation slice from TRAIN only (for early stopping)
Xa_tr, Xa_val, ya_tr, ya_val = train_test_split(
    Xa, ya, test_size=0.1, stratify=ya, random_state=42
)

# 7) Train LightGBM (AUC only, no class weighting)
gbm = lgb.LGBMClassifier(
    n_estimators=5000, learning_rate=0.03, num_leaves=31,
    min_child_samples=100, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.5, reg_lambda=5,
    metric='auc',
    random_state=42, n_jobs=-1, verbose=-1
)
t0 = time.time()
gbm.fit(
    Xa_tr, ya_tr, eval_set=[(Xa_val, ya_val)],
    callbacks=[lgb.early_stopping(100, first_metric_only=True),
               lgb.log_evaluation(0)]
)
train_time = time.time() - t0

# 8) Evaluate
tr = roc_auc_score(ya_tr, gbm.predict_proba(Xa_tr)[:, 1])
va = roc_auc_score(ya_val, gbm.predict_proba(Xa_val)[:, 1])
p = gbm.predict_proba(Xt)[:, 1]
te = roc_auc_score(yt, p)
fpr, tpr, _ = roc_curve(yt, p)

print("Train rows:", len(Xa), "| Test rows:", len(Xt))
print("Total input columns:", len(feat))
print("Trees used:", gbm.best_iteration_)
print(f"Training time: {train_time:.1f} sec")
print(f"Train AUC {tr:.4f} | Val AUC {va:.4f} | Test AUC {te:.4f} | "
      f"Gini {2*te-1:.4f} | KS {(tpr-fpr).max():.4f}")
print("Logistic was:  Test AUC 0.7516 | KS 0.3754")

# 9) Feature importance (share of total gain)
imp = pd.Series(gbm.booster_.feature_importance('gain'), index=feat)
imp = imp / imp.sum()
print("\nColumns the model actually used:", (imp > 0).sum(), "of", len(feat))
print("\nTop 20 features by gain:")
print(imp.sort_values(ascending=False).head(20).round(3))

## Part 5. XGBoost (self-contained)

Test AUC 0.7664, KS 0.3987. A tie with LightGBM; LightGBM trains about 2.6x faster.

In [ ]:
# ===== XGBoost on all columns + payment history (self-contained) =====
import time
import pandas as pd
import numpy as np
import xgboost as xgb
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

engine = create_engine("postgresql+psycopg2://postgres@localhost:5432/home_credit")

# 1) Same data, same cleanup, same split as LightGBM
full = pd.read_sql("SELECT * FROM staging_application_train", engine)
full['days_employed'] = full['days_employed'].replace(365243, np.nan)

idx_train, idx_test = train_test_split(
    full.index, test_size=0.2, stratify=full['target'], random_state=42
)

mv = pd.read_sql("SELECT * FROM mv_customer_payment_features", engine)
full = full.merge(mv, on='sk_id_curr', how='left')

for c in full.select_dtypes(include='object').columns:
    full[c] = full[c].astype('category')

feat = [c for c in full.columns if c not in ['sk_id_curr', 'target']]
Xa, ya = full.loc[idx_train, feat], full.loc[idx_train, 'target']
Xt, yt = full.loc[idx_test, feat], full.loc[idx_test, 'target']

Xa_tr, Xa_val, ya_tr, ya_val = train_test_split(
    Xa, ya, test_size=0.1, stratify=ya, random_state=42
)

# 2) Train XGBoost (AUC only, no class weighting, raw NaN)
xgbm = xgb.XGBClassifier(
    n_estimators=5000, learning_rate=0.03, max_depth=6,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.5, reg_lambda=5,
    tree_method='hist', enable_categorical=True,
    eval_metric='auc', early_stopping_rounds=100,
    random_state=42, n_jobs=-1
)
t0 = time.time()
xgbm.fit(Xa_tr, ya_tr, eval_set=[(Xa_val, ya_val)], verbose=False)
train_time = time.time() - t0

# 3) Evaluate
tr = roc_auc_score(ya_tr, xgbm.predict_proba(Xa_tr)[:, 1])
va = roc_auc_score(ya_val, xgbm.predict_proba(Xa_val)[:, 1])
p = xgbm.predict_proba(Xt)[:, 1]
te = roc_auc_score(yt, p)
fpr, tpr, _ = roc_curve(yt, p)
ks = (tpr - fpr).max()

print("Trees used:", xgbm.best_iteration + 1)
print(f"Training time: {train_time:.1f} sec")
print(f"Train AUC {tr:.4f} | Val AUC {va:.4f} | Test AUC {te:.4f} | Gini {2*te-1:.4f} | KS {ks:.4f}")

# 4) Full results log (earlier rows typed in from previous outputs, XGBoost row is live)
results = pd.DataFrame([
    ['1. Logistic, 19 feats (baseline)',        0.7444, 0.7460, 0.4921, 0.3678, None],
    ['2. Logistic, 19 feats + credit/goods',    0.7447, 0.7462, 0.4923, 0.3669, None],
    ['3. Logistic, 19 feats + payment hist.',   0.7511, 0.7516, 0.5032, 0.3754, None],
    ['4. Logistic, all cols + payment hist.',   0.7578, 0.7558, 0.5115, 0.3843, 30.0],
    ['5. LightGBM, all cols + payment hist.',   0.8446, 0.7678, 0.5355, 0.3982, 16.3],
    ['6. XGBoost, all cols + payment hist.',    tr,     te,     2*te-1, ks,     round(train_time, 1)],
], columns=['Model', 'Train AUC', 'Test AUC', 'Gini', 'KS', 'Train time (s)'])

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)
print("\n=== RESULTS LOG ===")
print(results.round(4).to_string(index=False))

results.round(4).to_csv("results_log.csv", index=False)
print("\nSaved to results_log.csv (in the notebook's working folder)")

# 5) Feature importance
imp = pd.Series(xgbm.get_booster().get_score(importance_type='gain'))
imp = (imp / imp.sum()).sort_values(ascending=False)
print("\nTop 20 features by gain:")
print(imp.head(20).round(3))


## Part 6. LightGBM vs XGBoost, importance on the same basis (self-contained)

Both use **total gain**. Rank agreement (Spearman) 0.98. An earlier version compared XGBoost average gain with LightGBM total gain, which was not comparable; the metrics were unaffected.

In [ ]:
# ===== LightGBM + XGBoost together, fair importance comparison (self-contained) =====
import time
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

engine = create_engine("postgresql+psycopg2://postgres@localhost:5432/home_credit")

# 1) Same data, cleanup and split as before
full = pd.read_sql("SELECT * FROM staging_application_train", engine)
full['days_employed'] = full['days_employed'].replace(365243, np.nan)

idx_train, idx_test = train_test_split(
    full.index, test_size=0.2, stratify=full['target'], random_state=42
)

mv = pd.read_sql("SELECT * FROM mv_customer_payment_features", engine)
full = full.merge(mv, on='sk_id_curr', how='left')

for c in full.select_dtypes(include='object').columns:
    full[c] = full[c].astype('category')

feat = [c for c in full.columns if c not in ['sk_id_curr', 'target']]
Xa, ya = full.loc[idx_train, feat], full.loc[idx_train, 'target']
Xt, yt = full.loc[idx_test, feat], full.loc[idx_test, 'target']

Xa_tr, Xa_val, ya_tr, ya_val = train_test_split(
    Xa, ya, test_size=0.1, stratify=ya, random_state=42
)

def evaluate(model, name, train_time):
    tr = roc_auc_score(ya_tr, model.predict_proba(Xa_tr)[:, 1])
    p = model.predict_proba(Xt)[:, 1]
    te = roc_auc_score(yt, p)
    fpr, tpr, _ = roc_curve(yt, p)
    ks = (tpr - fpr).max()
    print(f"{name}: Train AUC {tr:.4f} | Test AUC {te:.4f} | Gini {2*te-1:.4f} | "
          f"KS {ks:.4f} | {train_time:.1f} sec")
    return tr, te, ks

# 2) LightGBM (same settings as before)
gbm = lgb.LGBMClassifier(
    n_estimators=5000, learning_rate=0.03, num_leaves=31,
    min_child_samples=100, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.5, reg_lambda=5, metric='auc',
    random_state=42, n_jobs=-1, verbose=-1
)
t0 = time.time()
gbm.fit(Xa_tr, ya_tr, eval_set=[(Xa_val, ya_val)],
        callbacks=[lgb.early_stopping(100, first_metric_only=True), lgb.log_evaluation(0)])
t_lgb = time.time() - t0
lgb_res = evaluate(gbm, "LightGBM", t_lgb)

# 3) XGBoost (same settings as before)
xgbm = xgb.XGBClassifier(
    n_estimators=5000, learning_rate=0.03, max_depth=6,
    min_child_weight=5, subsample=0.8, colsample_bytree=0.5, reg_lambda=5,
    tree_method='hist', enable_categorical=True,
    eval_metric='auc', early_stopping_rounds=100,
    random_state=42, n_jobs=-1
)
t0 = time.time()
xgbm.fit(Xa_tr, ya_tr, eval_set=[(Xa_val, ya_val)], verbose=False)
t_xgb = time.time() - t0
xgb_res = evaluate(xgbm, "XGBoost ", t_xgb)

# 4) FAIR importance: TOTAL gain for both, each normalised to sum to 1
imp_lgb = pd.Series(gbm.booster_.feature_importance('gain'), index=feat)
imp_xgb = pd.Series(xgbm.get_booster().get_score(importance_type='total_gain')).reindex(feat).fillna(0)

imp = pd.DataFrame({
    'LightGBM': imp_lgb / imp_lgb.sum(),
    'XGBoost':  imp_xgb / imp_xgb.sum(),
})
imp['LGB rank'] = imp['LightGBM'].rank(ascending=False).astype(int)
imp['XGB rank'] = imp['XGBoost'].rank(ascending=False).astype(int)

print("\nTop 20 features (total gain, both models), sorted by LightGBM:")
print(imp.sort_values('LightGBM', ascending=False).head(20).round(3))

rank_corr = imp['LightGBM'].corr(imp['XGBoost'], method='spearman')
print(f"\nRank agreement between the two models (Spearman): {rank_corr:.2f}")

## Part 7. Tuning LightGBM with 5-fold CV on the train set only

**7a** is the grid behind the final model (8 configs, best CV AUC 0.7659 vs 0.7651 for the original, a tie). Final model: num_leaves=15, min_child_samples=200, colsample_bytree=0.3, reg_lambda=20, learning_rate=0.05. Test AUC 0.7677, KS 0.4034, train/test gap 0.053 (was 0.077).

**7b** is an earlier random search over 11 configs with `lgb.cv` (also tuning depth, `cat_smooth`, `min_data_per_group`). Best CV AUC 0.7659 vs 0.7650 for the original: the same conclusion. Optional.

In [ ]:
# ===== LightGBM tuning with 5-fold CV on TRAIN only (self-contained) =====
import time
import pandas as pd
import numpy as np
import lightgbm as lgb
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve

engine = create_engine("postgresql+psycopg2://postgres@localhost:5432/home_credit")

# 1) Same data, cleanup and split as every other model
full = pd.read_sql("SELECT * FROM staging_application_train", engine)
full['days_employed'] = full['days_employed'].replace(365243, np.nan)
idx_train, idx_test = train_test_split(
    full.index, test_size=0.2, stratify=full['target'], random_state=42
)
mv = pd.read_sql("SELECT * FROM mv_customer_payment_features", engine)
full = full.merge(mv, on='sk_id_curr', how='left')
for c in full.select_dtypes(include='object').columns:
    full[c] = full[c].astype('category')

feat = [c for c in full.columns if c not in ['sk_id_curr', 'target']]
Xa, ya = full.loc[idx_train, feat], full.loc[idx_train, 'target']
Xt, yt = full.loc[idx_test, feat], full.loc[idx_test, 'target']

# 2) Settings to compare (first = current setting, as the baseline)
base = dict(n_estimators=5000, learning_rate=0.05, subsample=0.8, subsample_freq=1,
            metric='auc', random_state=42, n_jobs=-1, verbose=-1)
configs = [
    dict(num_leaves=31, min_child_samples=100, colsample_bytree=0.5, reg_lambda=5),   # current
    dict(num_leaves=15, min_child_samples=100, colsample_bytree=0.5, reg_lambda=5),
    dict(num_leaves=15, min_child_samples=300, colsample_bytree=0.5, reg_lambda=10),
    dict(num_leaves=31, min_child_samples=300, colsample_bytree=0.5, reg_lambda=20),
    dict(num_leaves=7,  min_child_samples=200, colsample_bytree=0.4, reg_lambda=10),
    dict(num_leaves=15, min_child_samples=200, colsample_bytree=0.3, reg_lambda=20),
    dict(num_leaves=24, min_child_samples=400, colsample_bytree=0.4, reg_lambda=30),
    dict(num_leaves=10, min_child_samples=500, colsample_bytree=0.3, reg_lambda=50),
]

# 3) 5-fold CV on TRAIN only
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for i, cfg in enumerate(configs):
    t0 = time.time()
    aucs, trees = [], []
    for f_tr, f_va in skf.split(Xa, ya):
        m = lgb.LGBMClassifier(**base, **cfg)
        m.fit(Xa.iloc[f_tr], ya.iloc[f_tr],
              eval_set=[(Xa.iloc[f_va], ya.iloc[f_va])],
              callbacks=[lgb.early_stopping(100, first_metric_only=True),
                         lgb.log_evaluation(0)])
        aucs.append(m.best_score_['valid_0']['auc'])
        trees.append(m.best_iteration_)
    rows.append({'config': i, **cfg, 'CV AUC': np.mean(aucs), 'CV std': np.std(aucs),
                 'avg trees': int(np.mean(trees)), 'time (s)': round(time.time() - t0)})
    print(f"config {i} done | CV AUC {np.mean(aucs):.4f} ± {np.std(aucs):.4f} | {time.time()-t0:.0f}s")

cv = pd.DataFrame(rows).sort_values('CV AUC', ascending=False)
pd.set_option('display.width', 200); pd.set_option('display.max_columns', None)
print("\n=== CV RESULTS (train only, best first) ===")
print(cv.round(4).to_string(index=False))

# 4) Refit the best config, score TEST once
best = configs[int(cv.iloc[0]['config'])]
print("\nBest config:", best)
Xa_tr, Xa_val, ya_tr, ya_val = train_test_split(
    Xa, ya, test_size=0.1, stratify=ya, random_state=42
)
final = lgb.LGBMClassifier(**base, **best)
t0 = time.time()
final.fit(Xa_tr, ya_tr, eval_set=[(Xa_val, ya_val)],
          callbacks=[lgb.early_stopping(100, first_metric_only=True), lgb.log_evaluation(0)])
tt = time.time() - t0

tr = roc_auc_score(ya_tr, final.predict_proba(Xa_tr)[:, 1])
p = final.predict_proba(Xt)[:, 1]
te = roc_auc_score(yt, p)
fpr, tpr, _ = roc_curve(yt, p)
ks = (tpr - fpr).max()

print(f"\nTuned LightGBM: Trees {final.best_iteration_} | Train AUC {tr:.4f} | "
      f"Test AUC {te:.4f} | Gini {2*te-1:.4f} | KS {ks:.4f} | {tt:.1f} sec")
print(f"Train/test gap: {tr-te:.4f}  (untuned was 0.0768)")

# 5) Results log (earlier rows typed in, last row live)
results = pd.DataFrame([
    ['4. Logistic, all cols + payment hist.', 0.7578, 0.7558, 0.5115, 0.3843, 30.0],
    ['5. LightGBM untuned',                   0.8446, 0.7678, 0.5355, 0.3982, 16.3],
    ['6. XGBoost untuned',                    0.8614, 0.7664, 0.5328, 0.3987, 42.9],
    ['7. LightGBM tuned (CV)',                tr,     te,     2*te-1, ks,     round(tt, 1)],
], columns=['Model', 'Train AUC', 'Test AUC', 'Gini', 'KS', 'Train time (s)'])
print("\n=== RESULTS LOG ===")
print(results.round(4).to_string(index=False))
results.round(4).to_csv("results_log_tuning.csv", index=False)

In [ ]:
# ===== LightGBM tuning with 5-fold CV on TRAIN ONLY (self-contained) =====
import time, random
import pandas as pd
import numpy as np
import lightgbm as lgb
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split

engine = create_engine("postgresql+psycopg2://postgres@localhost:5432/home_credit")

# 1) Same data, cleanup, split as every other model
full = pd.read_sql("SELECT * FROM staging_application_train", engine)
full['days_employed'] = full['days_employed'].replace(365243, np.nan)
idx_train, idx_test = train_test_split(
    full.index, test_size=0.2, stratify=full['target'], random_state=42
)
mv = pd.read_sql("SELECT * FROM mv_customer_payment_features", engine)
full = full.merge(mv, on='sk_id_curr', how='left')
for c in full.select_dtypes(include='object').columns:
    full[c] = full[c].astype('category')

feat = [c for c in full.columns if c not in ['sk_id_curr', 'target']]
Xa, ya = full.loc[idx_train, feat], full.loc[idx_train, 'target']   # TRAIN only; test never used here

dtrain = lgb.Dataset(Xa, label=ya, free_raw_data=False)

# 2) Config 0 = current settings (for fair comparison), then 10 random configs
base = dict(num_leaves=31, min_child_samples=100, colsample_bytree=0.5,
            reg_lambda=5, max_depth=-1, min_split_gain=0.0, cat_smooth=10, min_data_per_group=100)

space = dict(
    num_leaves=[7, 15, 31],
    min_child_samples=[100, 200, 400, 800],
    colsample_bytree=[0.3, 0.5, 0.7],
    reg_lambda=[5, 20, 50, 100],
    max_depth=[-1, 4, 6],
    min_split_gain=[0.0, 0.1],
    cat_smooth=[10, 50, 200],          # shrinks rare categories toward the average
    min_data_per_group=[100, 300, 1000],
)
random.seed(42)
configs = [base]
while len(configs) < 11:
    c = {k: random.choice(v) for k, v in space.items()}
    if c not in configs:
        configs.append(c)

# 3) Run CV for each config
log = []
for i, cfg in enumerate(configs):
    params = dict(objective='binary', metric='auc', learning_rate=0.05,
                  subsample=0.8, subsample_freq=1, verbose=-1, n_jobs=-1,
                  seed=42, **cfg)
    t0 = time.time()
    res = lgb.cv(params, dtrain, num_boost_round=5000, nfold=5, stratified=True,
                 seed=42,
                 callbacks=[lgb.early_stopping(100, first_metric_only=True),
                            lgb.log_evaluation(0)])
    key_m = [k for k in res if k.endswith('auc-mean')][0]
    key_s = [k for k in res if k.endswith('auc-stdv')][0]
    n = len(res[key_m])
    row = dict(cfg_id=i, cv_auc=res[key_m][-1], cv_std=res[key_s][-1],
               trees=n, secs=round(time.time() - t0, 1), **cfg)
    log.append(row)
    print(f"cfg {i:2d} | CV AUC {row['cv_auc']:.4f} ± {row['cv_std']:.4f} | "
          f"trees {n:4d} | {row['secs']}s | {cfg}")

tune = pd.DataFrame(log).sort_values('cv_auc', ascending=False)
pd.set_option('display.width', 250); pd.set_option('display.max_columns', None)
print("\n=== TUNING RESULTS (best first) ===")
print(tune.round(4).to_string(index=False))
tune.round(4).to_csv("tuning_log.csv", index=False)
print("\nSaved to tuning_log.csv")
print(f"\nConfig 0 (current settings) CV AUC: {tune.loc[tune.cfg_id==0,'cv_auc'].values[0]:.4f}")
print(f"Best config CV AUC:                  {tune['cv_auc'].iloc[0]:.4f}")

## Part 8. SHAP on the tuned LightGBM (self-contained)

10,000-row test sample. Saves `shap_bar.png` and `shap_beeswarm.png`. `organization_type` is 3rd by gain but 9th by SHAP.

In [ ]:
# ===== SHAP on tuned LightGBM (self-contained) =====
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

engine = create_engine("postgresql+psycopg2://postgres@localhost:5432/home_credit")

# 1) Same data, cleanup and split as every other model
full = pd.read_sql("SELECT * FROM staging_application_train", engine)
full['days_employed'] = full['days_employed'].replace(365243, np.nan)
idx_train, idx_test = train_test_split(
    full.index, test_size=0.2, stratify=full['target'], random_state=42
)
mv = pd.read_sql("SELECT * FROM mv_customer_payment_features", engine)
full = full.merge(mv, on='sk_id_curr', how='left')
for c in full.select_dtypes(include='object').columns:
    full[c] = full[c].astype('category')

feat = [c for c in full.columns if c not in ['sk_id_curr', 'target']]
Xa, ya = full.loc[idx_train, feat], full.loc[idx_train, 'target']
Xt, yt = full.loc[idx_test, feat], full.loc[idx_test, 'target']
Xa_tr, Xa_val, ya_tr, ya_val = train_test_split(
    Xa, ya, test_size=0.1, stratify=ya, random_state=42
)

# 2) Retrain the tuned model (config 5)
model = lgb.LGBMClassifier(
    n_estimators=5000, learning_rate=0.05, subsample=0.8, subsample_freq=1,
    num_leaves=15, min_child_samples=200, colsample_bytree=0.3, reg_lambda=20,
    metric='auc', random_state=42, n_jobs=-1, verbose=-1
)
model.fit(Xa_tr, ya_tr, eval_set=[(Xa_val, ya_val)],
          callbacks=[lgb.early_stopping(100, first_metric_only=True), lgb.log_evaluation(0)])
print("Check: test AUC =", round(roc_auc_score(yt, model.predict_proba(Xt)[:, 1]), 4),
      "(should be about 0.7677)")

# 3) SHAP on a 10,000-row test sample
Xs = Xt.sample(10000, random_state=42)
explainer = shap.TreeExplainer(model)
sv = explainer.shap_values(Xs)
if isinstance(sv, list):          # some versions return [class0, class1]
    sv = sv[1]
sv = np.asarray(sv)

# 4) Mean |SHAP| vs gain importance
shap_imp = pd.Series(np.abs(sv).mean(axis=0), index=feat)
gain_imp = pd.Series(model.booster_.feature_importance('gain'), index=feat)
cmp = pd.DataFrame({
    'mean|SHAP|': shap_imp / shap_imp.sum(),
    'gain share': gain_imp / gain_imp.sum(),
})
cmp['SHAP rank'] = cmp['mean|SHAP|'].rank(ascending=False).astype(int)
cmp['gain rank'] = cmp['gain share'].rank(ascending=False).astype(int)

pd.set_option('display.width', 200)
print("\nTop 20 by mean |SHAP|:")
print(cmp.sort_values('mean|SHAP|', ascending=False).head(20).round(3))
print("\norganization_type ->")
print(cmp.loc[['organization_type']].round(3))
print("\nSpearman (SHAP vs gain):", round(cmp['mean|SHAP|'].corr(cmp['gain share'], method='spearman'), 2))

# 5) Plots (categoricals converted to codes only for colouring)
Xplot = Xs.copy()
for c in Xplot.select_dtypes('category').columns:
    Xplot[c] = Xplot[c].cat.codes.replace(-1, np.nan)

plt.figure()
shap.summary_plot(sv, Xplot, plot_type='bar', max_display=20, show=False)
plt.tight_layout(); plt.savefig("shap_bar.png", dpi=150); plt.show()

plt.figure()
shap.summary_plot(sv, Xplot, max_display=20, show=False)
plt.tight_layout(); plt.savefig("shap_beeswarm.png", dpi=150); plt.show()
print("Saved shap_bar.png and shap_beeswarm.png in the notebook folder")

## Part 9. Permutation importance vs gain vs SHAP (uses the untuned LightGBM)

Third opinion on feature importance. `organization_type`: gain rank 3, SHAP rank 9, permutation rank 14.

In [ ]:
# ===== SHAP + permutation importance for final LightGBM (self-contained) =====
import time
import pandas as pd
import numpy as np
import lightgbm as lgb
import shap
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

engine = create_engine("postgresql+psycopg2://postgres@localhost:5432/home_credit")

# 1) Same data, cleanup, split
full = pd.read_sql("SELECT * FROM staging_application_train", engine)
full['days_employed'] = full['days_employed'].replace(365243, np.nan)
idx_train, idx_test = train_test_split(
    full.index, test_size=0.2, stratify=full['target'], random_state=42
)
mv = pd.read_sql("SELECT * FROM mv_customer_payment_features", engine)
full = full.merge(mv, on='sk_id_curr', how='left')
for c in full.select_dtypes(include='object').columns:
    full[c] = full[c].astype('category')

feat = [c for c in full.columns if c not in ['sk_id_curr', 'target']]
Xa, ya = full.loc[idx_train, feat], full.loc[idx_train, 'target']
Xt, yt = full.loc[idx_test, feat], full.loc[idx_test, 'target']
Xa_tr, Xa_val, ya_tr, ya_val = train_test_split(
    Xa, ya, test_size=0.1, stratify=ya, random_state=42
)

# 2) Final LightGBM (same settings as before)
gbm = lgb.LGBMClassifier(
    n_estimators=5000, learning_rate=0.03, num_leaves=31,
    min_child_samples=100, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.5, reg_lambda=5, metric='auc',
    random_state=42, n_jobs=-1, verbose=-1
)
gbm.fit(Xa_tr, ya_tr, eval_set=[(Xa_val, ya_val)],
        callbacks=[lgb.early_stopping(100, first_metric_only=True), lgb.log_evaluation(0)])
base_auc = roc_auc_score(yt, gbm.predict_proba(Xt)[:, 1])
print(f"Test AUC (should be 0.7678): {base_auc:.4f}")

# 3) Split-gain importance (share of total gain)
gain = pd.Series(gbm.booster_.feature_importance('gain'), index=feat)
gain = gain / gain.sum()

# 4) SHAP on a random 5,000 test rows
t0 = time.time()
Xs = Xt.sample(5000, random_state=42)
explainer = shap.TreeExplainer(gbm)
sv = explainer.shap_values(Xs)
if isinstance(sv, list):          # older shap versions return [class0, class1]
    sv = sv[1]
sv = np.asarray(sv)
shap_imp = pd.Series(np.abs(sv).mean(axis=0), index=feat)
shap_imp = shap_imp / shap_imp.sum()
print(f"SHAP done in {time.time()-t0:.0f}s")

# 5) Permutation importance (AUC drop when a column is shuffled), top 30 by SHAP only
t0 = time.time()
Xp = Xt.sample(20000, random_state=1)
yp = yt.loc[Xp.index]
base_p = roc_auc_score(yp, gbm.predict_proba(Xp)[:, 1])
rng = np.random.RandomState(0)
perm = {}
for c in shap_imp.sort_values(ascending=False).head(30).index:
    drops = []
    for _ in range(3):
        Xshuf = Xp.copy()
        Xshuf[c] = Xshuf[c].values[rng.permutation(len(Xshuf))]
        drops.append(base_p - roc_auc_score(yp, gbm.predict_proba(Xshuf)[:, 1]))
    perm[c] = np.mean(drops)
perm = pd.Series(perm)
print(f"Permutation done in {time.time()-t0:.0f}s")

# 6) Side-by-side comparison
comp = pd.DataFrame({
    'gain_share': gain, 'shap_share': shap_imp,
    'gain_rank': gain.rank(ascending=False).astype(int),
    'shap_rank': shap_imp.rank(ascending=False).astype(int),
    'perm_AUC_drop': perm,
}).dropna(subset=['perm_AUC_drop'])
comp['perm_rank'] = comp['perm_AUC_drop'].rank(ascending=False).astype(int)
comp = comp.sort_values('shap_rank')
pd.set_option('display.width', 200)
print("\nTop 20 by SHAP (compare the three ranks):")
print(comp.head(20).round(4).to_string())

print("\nSpearman rank agreement (top-30 set):")
print(comp[['gain_rank', 'shap_rank', 'perm_rank']].corr(method='spearman').round(2))

# 7) Direction of effect for numeric features (correlation of value with SHAP)
print("\nDirection for top numeric features (+ = higher value raises risk):")
rows = []
for c in shap_imp.sort_values(ascending=False).head(15).index:
    if str(Xs[c].dtype) != 'category':
        j = feat.index(c)
        ok = Xs[c].notna()
        rows.append((c, round(np.corrcoef(Xs.loc[ok, c], sv[ok.values, j])[0, 1], 2)))
print(pd.DataFrame(rows, columns=['feature', 'corr(value, SHAP)']).to_string(index=False))

# 8) Beeswarm plot (wrapped: categorical columns can break some shap versions)
try:
    shap.summary_plot(sv, Xs, max_display=15, show=True)
except Exception as e:
    print("Beeswarm plot skipped:", e)

## Part 10. Decile table on the tuned LightGBM (self-contained)

Riskiest decile 28.4% default, safest 1.0%; riskiest 10% hold 35% of defaults, riskiest 30% hold 67%. Saves `decile_table.csv` and `decile_chart.png`.

In [ ]:
# ===== Decile table on tuned LightGBM (self-contained) =====
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

engine = create_engine("postgresql+psycopg2://postgres@localhost:5432/home_credit")

# 1) Same data, cleanup and split as every other model
full = pd.read_sql("SELECT * FROM staging_application_train", engine)
full['days_employed'] = full['days_employed'].replace(365243, np.nan)
idx_train, idx_test = train_test_split(
    full.index, test_size=0.2, stratify=full['target'], random_state=42
)
mv = pd.read_sql("SELECT * FROM mv_customer_payment_features", engine)
full = full.merge(mv, on='sk_id_curr', how='left')
for c in full.select_dtypes(include='object').columns:
    full[c] = full[c].astype('category')

feat = [c for c in full.columns if c not in ['sk_id_curr', 'target']]
Xa, ya = full.loc[idx_train, feat], full.loc[idx_train, 'target']
Xt, yt = full.loc[idx_test, feat], full.loc[idx_test, 'target']
Xa_tr, Xa_val, ya_tr, ya_val = train_test_split(
    Xa, ya, test_size=0.1, stratify=ya, random_state=42
)

# 2) Retrain tuned model (config 5)
model = lgb.LGBMClassifier(
    n_estimators=5000, learning_rate=0.05, subsample=0.8, subsample_freq=1,
    num_leaves=15, min_child_samples=200, colsample_bytree=0.3, reg_lambda=20,
    metric='auc', random_state=42, n_jobs=-1, verbose=-1
)
model.fit(Xa_tr, ya_tr, eval_set=[(Xa_val, ya_val)],
          callbacks=[lgb.early_stopping(100, first_metric_only=True), lgb.log_evaluation(0)])
p = model.predict_proba(Xt)[:, 1]
print("Check: test AUC =", round(roc_auc_score(yt, p), 4), "(should be about 0.7677)")

# 3) Deciles: 1 = riskiest 10%, 10 = safest 10%
d = pd.DataFrame({'p': p, 'y': yt.values})
d['decile'] = pd.qcut(d['p'].rank(method='first', ascending=False), 10, labels=range(1, 11))
tab = d.groupby('decile').agg(
    borrowers=('y', 'size'),
    defaults=('y', 'sum'),
    actual_default_rate=('y', 'mean'),
    avg_predicted_prob=('p', 'mean'),
)
overall = d['y'].mean()
tab['lift'] = tab['actual_default_rate'] / overall
tab['cum_defaults_caught'] = tab['defaults'].cumsum() / d['y'].sum()

pd.set_option('display.width', 200); pd.set_option('display.max_columns', None)
print(f"\nOverall default rate in test set: {overall:.4f}")
print(tab.round(4).to_string())
tab.round(4).to_csv("decile_table.csv")

# 4) Chart
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(tab.index.astype(int), tab['actual_default_rate'] * 100, color='#1f77b4', label='Actual')
ax.plot(tab.index.astype(int), tab['avg_predicted_prob'] * 100, 'o-', color='#d62728', label='Predicted')
ax.axhline(overall * 100, color='gray', ls='--', label='Overall rate')
ax.set_xlabel('Risk decile (1 = riskiest 10%)'); ax.set_ylabel('Default rate (%)')
ax.set_xticks(range(1, 11)); ax.legend()
plt.tight_layout(); plt.savefig("decile_chart.png", dpi=150); plt.show()
print("Saved decile_table.csv and decile_chart.png")

## Part 11. Calibration check (uses the UNTUNED LightGBM)

Raw model: average predicted 8.00% vs actual 8.07%, calibration slope 1.011, intercept 0.033, Brier 0.0670 vs 0.0742 for always predicting the base rate. Platt scaling and isotonic regression gave no improvement, so no recalibration was needed. **Not yet re-run on the tuned model.**

In [ ]:
# ===== Calibration check for final LightGBM (self-contained) =====
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss

engine = create_engine("postgresql+psycopg2://postgres@localhost:5432/home_credit")

# 1) Same data, cleanup, split
full = pd.read_sql("SELECT * FROM staging_application_train", engine)
full['days_employed'] = full['days_employed'].replace(365243, np.nan)
idx_train, idx_test = train_test_split(
    full.index, test_size=0.2, stratify=full['target'], random_state=42
)
mv = pd.read_sql("SELECT * FROM mv_customer_payment_features", engine)
full = full.merge(mv, on='sk_id_curr', how='left')
for c in full.select_dtypes(include='object').columns:
    full[c] = full[c].astype('category')

feat = [c for c in full.columns if c not in ['sk_id_curr', 'target']]
Xa, ya = full.loc[idx_train, feat], full.loc[idx_train, 'target']
Xt, yt = full.loc[idx_test, feat], full.loc[idx_test, 'target']
Xa_tr, Xa_val, ya_tr, ya_val = train_test_split(
    Xa, ya, test_size=0.1, stratify=ya, random_state=42
)

# 2) Final LightGBM
gbm = lgb.LGBMClassifier(
    n_estimators=5000, learning_rate=0.03, num_leaves=31,
    min_child_samples=100, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.5, reg_lambda=5, metric='auc',
    random_state=42, n_jobs=-1, verbose=-1
)
gbm.fit(Xa_tr, ya_tr, eval_set=[(Xa_val, ya_val)],
        callbacks=[lgb.early_stopping(100, first_metric_only=True), lgb.log_evaluation(0)])

eps = 1e-6
p_val = np.clip(gbm.predict_proba(Xa_val)[:, 1], eps, 1 - eps)
p_te  = np.clip(gbm.predict_proba(Xt)[:, 1], eps, 1 - eps)
y_te  = yt.values
base_rate = ya_tr.mean()
print(f"Test AUC (should be 0.7678): {roc_auc_score(y_te, p_te):.4f}")

# 3) Overall calibration on TEST (raw model)
logit = lambda p: np.log(p / (1 - p))
slope_fit = LogisticRegression(C=1e6, max_iter=1000).fit(logit(p_te).reshape(-1, 1), y_te)
print("\n=== RAW MODEL, TEST SET ===")
print(f"Average predicted risk: {p_te.mean()*100:.2f}%  |  Actual default rate: {y_te.mean()*100:.2f}%")
print(f"Brier score: {brier_score_loss(y_te, p_te):.4f}   (always-predict-{base_rate*100:.2f}%: "
      f"{brier_score_loss(y_te, np.full(len(y_te), base_rate)):.4f})")
print(f"Log loss:    {log_loss(y_te, p_te):.4f}   (always-predict-{base_rate*100:.2f}%: "
      f"{log_loss(y_te, np.full(len(y_te), base_rate)):.4f})")
print(f"Calibration slope: {slope_fit.coef_[0][0]:.3f} (ideal 1.0) | "
      f"intercept: {slope_fit.intercept_[0]:.3f} (ideal 0.0)")

# 4) Fixes, fit on VALIDATION slice only, judged on TEST
platt = LogisticRegression(C=1e6, max_iter=1000).fit(logit(p_val).reshape(-1, 1), ya_val)
p_platt = platt.predict_proba(logit(p_te).reshape(-1, 1))[:, 1]
iso = IsotonicRegression(out_of_bounds='clip', y_min=eps, y_max=1 - eps).fit(p_val, ya_val)
p_iso = iso.predict(p_te)

cmp = pd.DataFrame({
    'Version': ['Raw', 'Platt', 'Isotonic'],
    'Avg predicted %': [100*p.mean() for p in (p_te, p_platt, p_iso)],
    'Brier': [brier_score_loss(y_te, p) for p in (p_te, p_platt, p_iso)],
    'Log loss': [log_loss(y_te, np.clip(p, eps, 1-eps)) for p in (p_te, p_platt, p_iso)],
    'Test AUC': [roc_auc_score(y_te, p) for p in (p_te, p_platt, p_iso)],
})
print("\n=== RAW vs CALIBRATED (fit on validation slice, judged on test) ===")
print(cmp.round(4).to_string(index=False))

# 5) Decile table: predicted vs actual
df = pd.DataFrame({'raw': p_te, 'platt': p_platt, 'iso': p_iso, 'y': y_te})
df['decile'] = pd.qcut(df['raw'].rank(method='first'), 10, labels=range(1, 11)).astype(int)
g = df.groupby('decile').agg(customers=('y', 'size'), actual=('y', 'mean'),
                             raw=('raw', 'mean'), platt=('platt', 'mean'), iso=('iso', 'mean'))
g = (g * [1, 100, 100, 100, 100]).sort_index(ascending=False)
g.columns = ['customers', 'actual %', 'raw pred %', 'platt pred %', 'iso pred %']
print("\n=== PREDICTED vs ACTUAL BY DECILE (10 = riskiest) ===")
print(g.round(2).to_string())

# 6) Reliability plot
plt.figure(figsize=(5.5, 5.5))
gg = g.sort_index()
plt.plot(gg['raw pred %'], gg['actual %'], 'o-', label='Raw')
plt.plot(gg['platt pred %'], gg['actual %'], 's--', label='Platt')
mx = max(gg['raw pred %'].max(), gg['actual %'].max()) * 1.05
plt.plot([0, mx], [0, mx], 'k:', label='Perfect')
plt.xlabel('Average predicted default rate (%)'); plt.ylabel('Actual default rate (%)')
plt.title('Reliability by decile (test)'); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()